In [1]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets
import torch
from cnn import CharCNN, transform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_ds = datasets.EMNIST(
    root="./data",
    split="byclass",
    train=True,
    download=True,
    transform=transform
)

test_ds = datasets.EMNIST(
    root="./data",
    split="byclass",
    train=False,
    download=True,
    transform=transform
)

Device: cuda


100%|██████████| 562M/562M [00:02<00:00, 211MB/s]


In [2]:
num_classes = len(train_ds.classes)
print("Num classes:", num_classes)

batch_size = 256
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

Num classes: 62


In [3]:
model = CharCNN(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total

epochs = 15
best_acc = 0.0

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), "emnist_charcnn_best.pth")

    print(f"Epoch {epoch:02d}/{epochs} | "
          f"train loss {train_loss:.4f} acc {train_acc*100:.2f}% | "
          f"test loss {test_loss:.4f} acc {test_acc*100:.2f}% | "
          f"best {best_acc*100:.2f}%")

print("Saved best model to: emnist_charcnn_best.pth")

Epoch 01/15 | train loss 0.5212 acc 82.48% | test loss 0.3804 acc 86.02% | best 86.02%
Epoch 02/15 | train loss 0.3891 acc 85.88% | test loss 0.3604 acc 86.64% | best 86.64%
Epoch 03/15 | train loss 0.3649 acc 86.52% | test loss 0.3522 acc 86.82% | best 86.82%
Epoch 04/15 | train loss 0.3480 acc 87.01% | test loss 0.3477 acc 87.03% | best 87.03%
Epoch 05/15 | train loss 0.3347 acc 87.32% | test loss 0.3455 acc 87.15% | best 87.15%
Epoch 06/15 | train loss 0.3221 acc 87.68% | test loss 0.3484 acc 87.08% | best 87.15%
Epoch 07/15 | train loss 0.3115 acc 87.99% | test loss 0.3471 acc 87.14% | best 87.15%
Epoch 08/15 | train loss 0.3017 acc 88.28% | test loss 0.3500 acc 87.17% | best 87.17%
Epoch 09/15 | train loss 0.2919 acc 88.58% | test loss 0.3537 acc 87.06% | best 87.17%
Epoch 10/15 | train loss 0.2835 acc 88.82% | test loss 0.3619 acc 86.84% | best 87.17%
Epoch 11/15 | train loss 0.2753 acc 89.09% | test loss 0.3739 acc 86.99% | best 87.17%
Epoch 12/15 | train loss 0.2670 acc 89.39% 